In [2]:
import os
import time
import random
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

BASE_DIR = "output"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
FIG_DIR = os.path.join(BASE_DIR, "figures")
TABLE_DIR = os.path.join(BASE_DIR, "tables")
FINAL_DIR = os.path.join(BASE_DIR, "final")

for d in [BASE_DIR, CACHE_DIR, FIG_DIR, TABLE_DIR, FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

def sleep_polite(base=1.5, jitter=1.0):
    time.sleep(base + random.random() * jitter)

def get_sp500_tickers_fallback():
    return ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","JPM","XOM","UNH","PG","AVGO","TSLA","COST","HD","MA","BAC"]

def load_sp500_tickers():
    """
    Pulls S&P 500 tickers from Wikipedia. Falls back to a small list if unavailable.
    """
    try:
        tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
        df = tables[0]
        tickers = df["Symbol"].astype(str).str.replace(".", "-", regex=False).tolist()
        tickers = sorted(list(set(tickers)))
        return tickers
    except Exception:
        return get_sp500_tickers_fallback()

def download_prices(tickers, start="2019-01-01", end=None, chunk_size=80):
    """
    Downloads adjusted close prices via yf.download in batches.
    Caches each chunk to CSV to avoid repeated calls and rate-limit issues.
    """
    all_chunks = []
    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i+chunk_size]
        cache_path = os.path.join(CACHE_DIR, f"adjclose_{i}_{i+len(chunk)-1}_{start}.csv")

        if os.path.exists(cache_path):
            df_chunk = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        else:
            df = yf.download(
                tickers=chunk,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                group_by="column",
                threads=False
            )

            if df is None or df.empty:
                df_chunk = pd.DataFrame()
            else:
                if isinstance(df.columns, pd.MultiIndex):
                    # We used auto_adjust=True, so "Close" is adjusted close
                    df_chunk = df["Close"].copy()
                else:
                    df_chunk = df[["Close"]].rename(columns={"Close": chunk[0]})

            df_chunk.to_csv(cache_path)
            sleep_polite(base=2.0, jitter=1.5)

        all_chunks.append(df_chunk)

    prices = pd.concat(all_chunks, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()
    return prices

def compute_returns(prices):
    return prices.pct_change().dropna(how="all")

# Universe
tickers = load_sp500_tickers()
tickers = sorted(list(set(tickers)))

# Prices
prices = download_prices(tickers, start="2019-01-01")

# Market proxy
spy_prices = download_prices(["SPY"], start="2019-01-01").rename(columns={"SPY": "SPY"})

# Align on common trading days
prices = prices.join(spy_prices, how="inner")

# Returns
rets = compute_returns(prices)

# Save returns
rets_path = os.path.join(FINAL_DIR, "daily_returns.csv")
rets.to_csv(rets_path)

print(f"Saved daily returns to {rets_path}")
print("Tickers in returns panel (including SPY):", rets.shape[1])

# Report plots
plt.figure()
rets["SPY"].dropna().cumsum().plot()
plt.title("Cumulative SPY Return (Daily, Simple Sum Approximation)")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "spy_cumulative_return.png"), dpi=200)
plt.close()

coverage = rets.drop(columns=["SPY"]).notna().sum(axis=1)
plt.figure()
coverage.plot()
plt.title("Daily Data Coverage: Number of Stocks with Returns Available")
plt.xlabel("Date")
plt.ylabel("Count of Stocks")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "coverage_count_over_time.png"), dpi=200)
plt.close()

# Summary table
summary = pd.DataFrame({
    "start_date": [rets.index.min()],
    "end_date": [rets.index.max()],
    "n_days": [len(rets.index)],
    "n_stocks_excl_spy": [rets.shape[1] - 1]
})
summary.to_csv(os.path.join(TABLE_DIR, "sample_summary.csv"), index=False)
print("Saved sample summary table to output/tables/sample_summary.csv")

Saved daily returns to output/final/daily_returns.csv
Tickers in returns panel (including SPY): 17
Saved sample summary table to output/tables/sample_summary.csv
